# Exercise 01 – Signals, noise, and channel capacity

An interactive model of a harmonic signal corrupted by Gaussian noise. The signal-to-noise ratio and the channel capacity are computed with functions that you implement yourself; two further dashboards explore the Shannon–Hartley theorem and a telegraph link over a band-limited, noisy channel, and the model then serves for experiments with real audio and image data. Theory and task descriptions are in the [README](../README.md).

## Imports

In [1]:
from pathlib import Path

import matplotlib.image as image
import matplotlib.pyplot as plt
import numpy as np
import panel as pn
from IPython.display import Audio, display
from scipy.io import wavfile

from lib.core import CapacityExplorer, HarmSignal, NoiseSignal, Telegraph, card_html, si_format
from lib.params import add_logo, rc_params

pn.extension()
plt.rcParams.update(rc_params)

## Constants

In [2]:
rng = np.random.default_rng()

NO_SAMPLES = 1000  # samples per signal over the time axis 0–π s
AUDIO_PATH = "input.wav"  # mono 16-bit WAV in the notebook directory
IMAGE_PATH = "../fig/android_gray.jpeg"

## Theory

### A harmonic signal and Gaussian noise

`HarmSignal` generates $y(t) = A \sin(2 \pi f t + \varphi)$ on $t \in [0, \pi]$ s; `NoiseSignal` draws independent samples from the normal distribution with zero mean and the standard deviation $\sigma$ set by its slider, so its power is $\sigma^2$. Adding the two objects links them: the combined signal follows every change made with the sliders.

In [3]:
signal = HarmSignal(f=np.sin, amplitude=1, freq=5, band=10, no_samples=NO_SAMPLES, title="Signal")
noise = NoiseSignal(f=rng.normal, sigma=0.1, no_samples=NO_SAMPLES, title="Noise")
combined = signal + noise

## Exercise

### Task 1 – Implement the channel metrics

1. Implement `calc_signal_power(signal)` returning the mean power of a sampled signal.
2. Implement `calc_channel_capacity(S, N, B)` returning the Shannon–Hartley capacity for signal power `S`, noise power `N`, and bandwidth `B`.
3. Run the dashboard in Task 2 and verify the displayed values against a manual computation for one slider setting.

Both formulas are in the README. Keep the signatures below; the dashboard calls them by name.

In [15]:
def calc_signal_power(signal: np.ndarray) -> float:
    """Mean power of a sampled signal [W]."""
    signal_power = np.nan
    return signal_power


def calc_channel_capacity(S: float, N: float, B: float) -> float:
    """Channel capacity [bit/s] for signal power S [W], noise power N [W], and bandwidth B [Hz]."""
    channel_capacity = np.nan
    return channel_capacity

### Task 2 – Model a signal with noise

1. Before touching anything, predict qualitatively how the signal-to-noise ratio and the channel capacity react to each slider.
2. Vary one parameter at a time and press *Calculate*: the button reads the current signal and noise and shows the powers, the SNR (linear and in dB), and the capacity computed by your functions.
3. Record the parameter values, the results, and whether they support the prediction.

The noise power is measured over the whole sampled band, so the *Channel bandwidth* slider does not filter anything; it is only the factor $B$ of the capacity formula. The consequence is that the SNR stays fixed while $B$ changes, an assumption discussed in the README. The channel is a band-pass of width $B$ centred on the tone, so it occupies $f - B/2$ to $f + B/2$ and needs $f \geq B/2$ to fit above 0 Hz; the *Bandwidth* slider therefore moves the lower limit of the *Frequency* slider and raises the tone when it would fall below $B/2$.

In [5]:
result = pn.pane.HTML(card_html("Channel metrics", [("Press Calculate", "")]))


def calculate(event):
    S = calc_signal_power(signal.y)
    N = calc_signal_power(noise.y)
    B = signal.band.value
    snr = S / N
    result.object = card_html(
        "Channel metrics",
        [
            ("Signal power S", f"{S:.4g} W"),
            ("Noise power N", f"{N:.4g} W"),
            ("SNR", f"{snr:.4g} [–] = {10 * np.log10(snr):.1f} dB"),
            ("Bandwidth B", f"{B:g} Hz"),
            ("Capacity C", si_format(calc_channel_capacity(S, N, B), "bit/s")),
        ],
        note="Powers are means of the squared samples across a 1 Ω load.",
    )


button = pn.widgets.Button(name="Calculate", button_type="primary")
button.on_click(calculate)

pn.Column(
    pn.Row(
        pn.Column(signal.amplitude, signal.freq, signal.phase, signal.band),
        signal.plot,
        combined.plot,
    ),
    pn.Row(
        pn.Column(noise.sigma),
        noise.plot,
        pn.Column(pn.Spacer(height=20), button, result),
    ),
)

Column
    [0] Row
        [0] Column
            [0] Bokeh(Slider)
            [1] Bokeh(Slider)
            [2] Bokeh(Slider)
            [3] Bokeh(Slider)
        [1] Bokeh(figure)
        [2] Bokeh(figure)
    [1] Row
        [0] Column
            [0] Bokeh(Slider)
        [1] Bokeh(figure)
        [2] Column
            [0] Spacer(height=20)
            [1] Button(button_type='primary', color='primary', label='Calculate', name='Calculate')
            [2] HTML(str)

### Reference implementation

Run this cell only after Task 1 is complete. It replaces your functions with the reference versions from `lib/core.py`; press *Calculate* again and compare.

In [6]:
from lib.core import calc_channel_capacity, calc_signal_power

### Task 3 – Explore the Shannon–Hartley theorem

The explorer plots $C = B \log_2(1 + S/N)$ in two ways. The left plot shows the capacity as a function of bandwidth at the chosen SNR on logarithmic axes, with the SNR = 0 dB and 30 dB cases as grey references. The right plot shows the spectral efficiency $C/B$ as a function of SNR; this curve does not depend on $B$ at all, and the marker only moves along it. The blue marker is the operating point on both plots.

1. Select each technology preset from the README table and note $C$ and $C/B$. Which technologies reach their rate mainly by bandwidth, and which mainly by SNR?
2. With a preset of your choice, double the bandwidth (move the slider by 0.3 on the $\log_{10}$ scale) and then double the SNR (add 3 dB). Compare the two gains.
3. Find the SNR below which the high-SNR approximation $C/B \approx \log_2(S/N)$ fails by more than 10 %.

In [7]:
explorer = CapacityExplorer()

pn.Column(
    pn.Row(explorer.preset, explorer.log_band, explorer.snr_db),
    pn.Row(explorer.plot_band, explorer.plot_snr),
    explorer.card,
)

Column
    [0] Row
        [0] Bokeh(Select)
        [1] Bokeh(Slider)
        [2] Bokeh(Slider)
    [1] Row
        [0] Bokeh(figure)
        [1] Bokeh(figure)
    [2] Bokeh(Div)

### Task 4 – Telegraph over a band-limited, noisy channel

A binary telegraph sends 32 bits as rectangular on–off pulses at the symbol rate $R_s$. The channel keeps only frequencies up to $B$, either with an ideal (brick-wall) low-pass filter or with a first-order RC low-pass that models a long telegraph line, and then adds Gaussian noise. The receiver samples the middle of every symbol interval and decides against a 0.5 V threshold; correct decisions are teal dots, errors red crosses.

The third plot shows the magnitude characteristic $|H(f)|$ of the selected channel model (dashed) together with the line spectra of the transmitted and received waveforms, all in dB relative to the strongest transmitted component; the dotted line marks $B$. The spectra are computed over exactly the 32-symbol block, so they are the Fourier series of the pattern with lines spaced $R_s/32$: the *Alternating* pattern has lines only at DC and the odd multiples of $R_s/2$, *Isolated ones* at the multiples of $R_s/4$, and the *Random* pattern fills every line of the periodically repeated block. The received spectrum is the steady-state response of the periodic pattern; the grey band in the received-signal plot marks the filter transient ($5\tau$ after the block starts for the RC line, $1/B$ of ringing at both edges for the ideal low-pass), where the waveform has not yet settled to that steady state.

**Why the RC line produces ramps, not sines.** The Nyquist argument that a channel "strips the higher frequencies" assumes a brick-wall filter. The *Ideal low-pass* does exactly that: for the alternating pattern, a square wave with fundamental $R_s/2$ and odd harmonics at $3R_s/2, 5R_s/2, \dots$ of amplitudes $1, 1/3, 1/5, \dots$, choosing $B$ between $R_s/2$ and $3R_s/2$ keeps a single harmonic and the received signal is a clean sine. The *RC line* has no cut-off, only a slope: $H(f) = 1/(1 + \mathrm{j} f/B)$ falls by 6 dB per octave, so nothing is removed and every harmonic is merely attenuated by about $B/f$ once $f \gg B$. The harmonic amplitudes then go as $\frac{1}{n} \cdot \frac{B}{n f_0} \propto 1/n^2$, which is the spectrum of a triangle wave, not a sine. In the time domain the same thing appears as the step response of an RC circuit with $\tau = RC = 1/(2 \pi B)$: every edge starts an exponential $1 - e^{-t/\tau}$ and the received pulses become the "shark fins" seen in the plot. At $R_s \ll 2B$ the exponential settles within the symbol and the pulses stay nearly rectangular; at $R_s \approx 2B$ it no longer settles and the fins fall short of 0 V and 1 V, shrinking the decision margin; at $R_s \gg 2B$ only a small triangle around 0.5 V remains, but its harmonics still arrive at $1/n^2$ and it never turns into a sine. Compare the two models in the spectrum plot: the ideal filter deletes the fundamental at once, giving a sharp limit at $2B$, whereas the RC line only tilts the spectrum, so the link degrades gradually as $\tau$ grows relative to $T_s$.

1. Keep $\sigma = 0$ and the *Ideal low-pass* model. For the *Alternating* pattern, raise $R_s$ until errors appear and compare the value with $2B$; repeat for two other bandwidths. Explain why the *Isolated ones* pattern survives to a higher symbol rate.
2. Switch to the *RC line* model and repeat. Which model has a sharp limit and which degrades gradually? Relate the shape of the received pulses to the step response of an RC circuit.
3. Return to a setting without errors, then raise $\sigma$ until errors appear. Note how the margin left by the filter determines how much noise the link tolerates; compare a setting with $R_s \ll 2B$ and a setting with $R_s$ close to $2B$.
4. For an error-free setting, the raw bit rate is $R_s$ bit/s (one bit per symbol). Compare it with the Shannon–Hartley capacity for the same $B$ and an SNR computed from the pulse amplitude and $\sigma$.

In [8]:
telegraph = Telegraph()

pn.Column(
    pn.Row(
        pn.Column(telegraph.pattern, telegraph.filter),
        pn.Column(telegraph.symbol_rate, telegraph.band, telegraph.sigma),
    ),
    pn.Row(telegraph.plot_tx, telegraph.plot_rx),
    pn.Row(telegraph.plot_spectrum, pn.Column(pn.Spacer(height=20), telegraph.card)),
)

Column
    [0] Row
        [0] Column
            [0] Bokeh(Select)
            [1] Bokeh(Select)
        [1] Column
            [0] Bokeh(Slider)
            [1] Bokeh(Slider)
            [2] Bokeh(Slider)
    [1] Row
        [0] Bokeh(figure)
        [1] Bokeh(figure)
    [2] Row
        [0] Bokeh(figure)
        [1] Column
            [0] Spacer(height=20)
            [1] Bokeh(Div)

### Task 5 – Apply noise to audio

1. Record or generate a short speech signal and save it as a mono 16-bit WAV file named `input.wav` next to this notebook (`ffmpeg -i recording.m4a -ar 8000 -ac 1 -sample_fmt s16 input.wav` converts most formats). Without the file, the template falls back to a synthetic vowel-like tone.
2. Add Gaussian noise of increasing standard deviation and listen to each version at a consistent, comfortable playback level.
3. Compute the SNR of each version with your `calc_signal_power` and relate it to what you hear.

The template loads the file, adds noise of one strength, plays the result, and prints the SNR; extend it to a series of noise levels.

In [14]:
if Path(AUDIO_PATH).exists():
    fs, audio = wavfile.read(AUDIO_PATH)
    audio = audio.astype(np.float64) / 32768  # int16 → [-1, 1)
else:
    print(f"{AUDIO_PATH} not found; using a synthetic vowel-like tone instead.")
    fs = 8000
    t = np.arange(0, 2, 1 / fs)
    harmonics = [(150, 1.0), (300, 0.6), (450, 0.4), (600, 0.3)]  # 150 Hz pitch
    audio = sum(a * np.sin(2 * np.pi * f * t) for f, a in harmonics)
    audio *= 0.5 * (1 - np.cos(2 * np.pi * 3 * t))  # three syllable-like bursts
    audio *= 0.5 / np.abs(audio).max()

sigma = 0.01
noisy = audio + rng.normal(0, sigma, audio.shape)

snr = calc_signal_power(audio) / calc_signal_power(noisy - audio)
print(f"SNR = {snr:.1f} [–] = {10 * np.log10(snr):.1f} dB")
display(Audio(audio, rate=fs))
display(Audio(noisy, rate=fs))

input.wav not found; using a synthetic vowel-like tone instead.
SNR = 258.5 [–] = 24.1 dB


### Task 6 – Apply noise to an image

1. Load the grayscale image, scale it to $[0, 1]$, and add Gaussian noise with $\sigma = 15, 25$, and $50$ on the 8-bit scale, i.e. $\sigma = 15/255, 25/255, 50/255$. These are the noise levels used in the image-denoising literature, so the results can be compared with published figures.
2. Display the original and the three noisy versions side by side.
3. Compute the SNR of each version before and after clipping. The same image returns in Exercise 04, where the degradation is measured with PSNR and SSIM.

Values outside $[0, 1]$ must be clipped before display; clipping changes the error, so distinguish the generated noise from the error remaining in the saved image. The template keeps both: `noisy_img` is the unclipped sum, whose error is exactly the generated noise, and `clipped_img` is the displayable version, whose error is smaller because clipping pulls the out-of-range pixels back toward the original. Explain why the SNR after clipping is always at least as high, and why the difference grows with $\sigma$.

> **Note on clipping.** Additive Gaussian noise followed by clipping to the valid range is the standard convention (MATLAB `imnoise`, scikit-image `random_noise`), but it is only a good approximation of additive noise when few pixels are near black or white. This image has about 30 % of its pixels below 0.05, so even at $\sigma = 15/255$ roughly 15 % of the pixels are clipped, and the remaining error is no longer Gaussian nor zero-mean in the dark regions. On a mid-tone test image such as *cameraman* the same $\sigma$ clips only 1–2 %. Keep this in mind when comparing the SNR before and after clipping, and later the PSNR in Exercise 04, with published values.

In [ ]:
img = image.imread(IMAGE_PATH).astype(np.float64) / 255

sigmas = [15 / 255, 25 / 255, 50 / 255]  # noise levels of the denoising benchmarks

fig, axes = plt.subplots(1, len(sigmas) + 1, figsize=(16, 4))
fig.subplots_adjust(top=0.78)  # room for the two-line titles below the logo
add_logo(fig)
axes[0].imshow(img, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original", fontsize=15)
for ax, sigma in zip(axes[1:], sigmas):
    noise = rng.normal(0, sigma, img.shape)
    noisy_img = img + noise  # what the noise model produces; values leave [0, 1]
    clipped_img = np.clip(noisy_img, 0, 1)  # what can be displayed and saved
    snr_noise = calc_signal_power(img) / calc_signal_power(noise)
    snr_clipped = calc_signal_power(img) / calc_signal_power(clipped_img - img)
    print(
        f"sigma {sigma * 255:.0f}/255: {np.mean((noisy_img < 0) | (noisy_img > 1)):.1%} of pixels clipped, "
        f"SNR {10 * np.log10(snr_noise):.1f} dB before clipping, "
        f"{10 * np.log10(snr_clipped):.1f} dB after"
    )
    ax.imshow(clipped_img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        f"σ = {sigma * 255:.0f}/255\nSNR {10 * np.log10(snr_noise):.1f} dB → {10 * np.log10(snr_clipped):.1f} dB clipped",
        fontsize=15,
    )
for ax in axes:
    ax.set_axis_off()
plt.show()

## Questions

1. What is the unit in which SNR is commonly expressed, and how does it relate to the linear ratio used in the Shannon–Hartley formula?
2. **Extension:** How can independent spatial streams increase total capacity while sharing the same frequency band? Why is antenna count alone insufficient to predict the gain?
3. Compute the capacity of the analog voiceband telephone channel in the README table. The 64 kbit/s PCM stream represents the sampled speech and travels over a digital network connection. Why does comparing these two rates not show a violation of the channel-capacity bound?
4. What happens to a 5 kHz tone sampled at $f_s = 8$ kHz?
5. In the telegraph simulation, why is the alternating pattern 0101… the first to fail when the symbol rate exceeds $2B$, and what does this have in common with the sampling theorem?
6. The telegraph link with $\sigma = 0$ and $R_s < 2B$ transmits without errors, yet the Shannon–Hartley capacity for $N = 0$ is infinite. Which assumption of the simulation, rather than of the theorem, keeps the rate finite?